**1. CONVOLUTIONAL AUTOENCODERS TRAINING**

In [ ]:
import digitalhub as dh
import pandas as pd
import matplotlib.pyplot as plt

NOME_PROGETTO = "floods"
project = dh.get_project(NOME_PROGETTO)
print(f"Progetto: {project.name}")

**SETUP ENVIRONMENT**

In [37]:
# PARAMETRI JOB
job_name = "train_s1_v1"    # nome da passare poi a notebook moco 
dataset = "Standard"            # Test, Standard, Anomalies*

encoders_train_func = project.new_function(
    name= f'encoders-Floods_{dataset}-{job_name}',
    kind="python",
    python_version="PYTHON3_10",
    code_src="../src/", 
    handler="pretrain_encoder_s1", 
    base_image="pytorch/pytorch:2.1.2-cuda11.8-cudnn8-runtime",
    requirements=["pandas==2.3.3", "numpy==1.26.4", "rasterio==1.4.4", "tqdm==4.70.0", "tifffile==2024.8.30"]
)

volumi = [
    {
        "volume_type": "ephemeral",
        "name": "volume-spazio-dati",
        "mount_path": "/data",      
        "spec": {"size": "500Gi"}   
    }
]

# .run("build") -> scarica immagine, installa requirements, copia intera cartella code_src, crea immagine docker

build = encoders_train_func.run("build", wait=True)
print(f"BUILD: {build.status.state}")


2026-08-19 14:56:34,821 - dhcore.digitalhub.entities.run._base.entity - INFO - Waiting for run af1c2d46fb7b4ef788c6e80de2a2cff2 to finish...
2026-08-19 14:56:39,832 - dhcore.digitalhub.entities.run._base.entity - INFO - Waiting for run af1c2d46fb7b4ef788c6e80de2a2cff2 to finish...
2026-08-19 14:56:44,845 - dhcore.digitalhub.entities.run._base.entity - INFO - Waiting for run af1c2d46fb7b4ef788c6e80de2a2cff2 to finish...
2026-08-19 14:56:49,858 - dhcore.digitalhub.entities.run._base.entity - INFO - Waiting for run af1c2d46fb7b4ef788c6e80de2a2cff2 to finish...
2026-08-19 14:56:54,975 - dhcore.digitalhub.entities.run._base.entity - INFO - Waiting for run af1c2d46fb7b4ef788c6e80de2a2cff2 to finish...
2026-08-19 14:56:59,988 - dhcore.digitalhub.entities.run._base.entity - INFO - Waiting for run af1c2d46fb7b4ef788c6e80de2a2cff2 to finish...
2026-08-19 14:57:05,002 - dhcore.digitalhub.entities.run._base.entity - INFO - Waiting for run af1c2d46fb7b4ef788c6e80de2a2cff2 to finish...
2026-08-19 14

BUILD: COMPLETED


**TRAINING**

In [ ]:
# time_debug = True solo per debug, = False per training

parametri = {
    "epochs": 200, 
    "batch_size": 16, 
    "lr": 1e-4, 
    "weight_decay": 1e-4,      
    "patch_size": 256, 
    "n_images1": 4, "n_channels1": 2,                       # sar
    "n_images2": 4, "n_channels2": 10,                      # ottiche               
    "mamba": False, 
    "workers": 0,
    "patience": 20,
    "job_name": job_name,
    "dataset": dataset,
    "time_debug": False
}

print(f"PARAMETRI: {parametri}")

# action job = avvia container, esegui script, libera risorse

run_train_encoders = encoders_train_func.run(
    action="job", 
    parameters=parametri, 
    volumes=volumi, 
    profile="1xV100",                                       # 1x = 1 gpu
    # local_execution= True,                                
    wait=True
)

print(f"Run train_encoders avviato: {run_train_encoders.id}")

PARAMETRI: {'epochs': 200, 'batch_size': 16, 'lr': 0.0001, 'weight_decay': 0.0001, 'patch_size': 256, 'n_images1': 4, 'n_channels1': 2, 'n_images2': 4, 'n_channels2': 10, 'mamba': False, 'workers': 0, 'patience': 20, 'job_name': 'train_s1_v1', 'dataset': 'Standard', 'time_debug': False}


2026-08-19 14:57:25,444 - dhcore.digitalhub.entities.run._base.entity - INFO - Waiting for run 860447a17a0641d480ea889d2942ecdb to finish...
2026-08-19 14:57:30,454 - dhcore.digitalhub.entities.run._base.entity - INFO - Waiting for run 860447a17a0641d480ea889d2942ecdb to finish...
2026-08-19 14:57:35,467 - dhcore.digitalhub.entities.run._base.entity - INFO - Waiting for run 860447a17a0641d480ea889d2942ecdb to finish...
2026-08-19 14:57:40,593 - dhcore.digitalhub.entities.run._base.entity - INFO - Waiting for run 860447a17a0641d480ea889d2942ecdb to finish...
2026-08-19 14:57:45,605 - dhcore.digitalhub.entities.run._base.entity - INFO - Waiting for run 860447a17a0641d480ea889d2942ecdb to finish...
2026-08-19 14:57:50,619 - dhcore.digitalhub.entities.run._base.entity - INFO - Waiting for run 860447a17a0641d480ea889d2942ecdb to finish...
2026-08-19 14:57:55,633 - dhcore.digitalhub.entities.run._base.entity - INFO - Waiting for run 860447a17a0641d480ea889d2942ecdb to finish...
2026-08-19 14

In [ ]:
# rieseguibile
run_train_encoders.refresh()    
print(run_train_encoders.status.state)
print(run_train_encoders.status.message)
# print(run_train_encoders.logs())

COMPLETED
None


**PLOTS**

In [ ]:
# salvataggio log
path_s1 = project.get_artifact(f"metrics-s1_{job_name}").download(overwrite=True)
path_s2 = project.get_artifact(f"metrics-s2_{job_name}").download(overwrite=True)

df_s1 = pd.read_csv(path_s1)
df_s2 = pd.read_csv(path_s2)

# plots
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))
plt.title(f"Training Encoders - {job_name}")

# sar
ax1.plot(df_s1['epoch'], df_s1['train_loss'], color='blue', label='Train Loss')
ax1.set_title(f'SAR')
ax1.set_xlabel('Epochs')
ax1.set_ylabel('Loss (MSE)')
ax1.grid(True)

# ottico
ax2.plot(df_s2['epoch'], df_s2['train_loss'], color='red', label='Train Loss')
ax2.set_title(f'OTTICO')
ax2.set_xlabel('Epochs')
ax2.grid(True)

plt.show()


BackendError: No object found.